In [19]:
%pip install catboost

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/100.2 MB 1.5 MB/s eta 0:01:06
   ---------------------------------------- 1.0/100.2 MB 1.9 MB/s eta 0:00:52
    --------------------------------------- 1.3/100.2 MB 1.9 MB/s eta 0:00:54
    --------------------------------------- 1.8/100.2 MB 2.0 MB/s eta 0:00:49
    --------------------------------------- 2.1/100.2 MB 1.9 MB/s eta 0:00:53
   - -------------------------------------- 2.6/100.2 MB 1.8 MB/s eta 0:00:54
   - -------------------------------------- 3.1/100.2 MB 1.9 MB/s eta 0:00:50
   - -------------------------------------- 3.4/100.2 MB 1.8 MB/s eta 0:00:53
   - -------------------------------------- 3.7/100.2 MB 1.8 MB/s eta 0:00:53
   - -------------------------------------- 4.2/100.2 MB 1.9 MB/s eta 0:00:51
   - -------------------------------------- 4.2/100.2 MB 1.9 MB/s eta 0:00:51



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: C:\Users\MAN GHADIYA\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


In [47]:
# ===============================
# 1. Import Libraries
# ===============================
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [28]:
# ===============================
# 2. Load Dataset
# ===============================
df = pd.read_csv("f2.csv")   # change filename

df.head()

,Temparature,Humidity,Moisture,Soil_Type,Crop_Type,Nitrogen,Potassium,Phosphorous,Fertilizer
0,20,83,26,Clayey,rice,90,49,36,Urea
1,25,84,32,Loamy,rice,66,59,36,Urea
2,33,64,50,Loamy,Wheat,41,0,0,Urea
3,34,65,54,Loamy,Wheat,38,0,0,Urea
4,38,72,51,Loamy,Wheat,39,0,0,Urea


In [29]:
print(df.shape)
print(df.columns)
print(df.isnull().sum())

(552, 9)
Index(['Temparature', 'Humidity', 'Moisture', 'Soil_Type', 'Crop_Type',
       'Nitrogen', 'Potassium', 'Phosphorous', 'Fertilizer'],
      dtype='object')
Temparature    0
Humidity       0
Moisture       0
Soil_Type      0
Crop_Type      0
Nitrogen       0
Potassium      0
Phosphorous    0
Fertilizer     0
dtype: int64


In [30]:
# ===============================
# 3. Clean Column Names
# ===============================
df.columns = df.columns.str.strip()

# rename spelling mistakes
df = df.rename(columns={
    "Temparature": "Temperature",
    "Phosphorous": "Phosphorus"
})

df.columns

Index(['Temperature', 'Humidity', 'Moisture', 'Soil_Type', 'Crop_Type',
       'Nitrogen', 'Potassium', 'Phosphorus', 'Fertilizer'],
      dtype='object')

In [32]:
# ===============================
# 4. Basic Info
# ===============================
print("Shape:", df.shape)
print("\nMissing Values:\n")
print(df.isnull().sum())

print("\nUnique Fertilizers:\n")
print(df["Fertilizer"].value_counts())

Shape: (552, 9)

Missing Values:

Temperature    0
Humidity       0
Moisture       0
Soil_Type      0
Crop_Type      0
Nitrogen       0
Potassium      0
Phosphorus     0
Fertilizer     0
dtype: int64

Unique Fertilizers:

Fertilizer
Urea                  108
DAP                   104
28-28                  68
20-20                  56
14-35-14               56
TSP                    28
10-26-26               28
17-17-17               28
14-14-14               16
15-15-15               16
10-10-10               16
Superphosphate         12
Potassium sulfate.     12
Potassium chloride      4
Name: count, dtype: int64


In [33]:
# ===============================
# 5. Features / Target
# ===============================
X = df.drop("Fertilizer", axis=1)
y = df["Fertilizer"]

X.head()

,Temperature,Humidity,Moisture,Soil_Type,Crop_Type,Nitrogen,Potassium,Phosphorus
0,20,83,26,Clayey,rice,90,49,36
1,25,84,32,Loamy,rice,66,59,36
2,33,64,50,Loamy,Wheat,41,0,0
3,34,65,54,Loamy,Wheat,38,0,0
4,38,72,51,Loamy,Wheat,39,0,0


In [34]:
# ===============================
# 6. Define Columns
# ===============================
categorical_cols = ["Soil_Type", "Crop_Type"]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

print("Categorical:", categorical_cols)
print("Numeric:", numeric_cols)

Categorical: ['Soil_Type', 'Crop_Type']
Numeric: ['Temperature', 'Humidity', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorus']


In [35]:
# ===============================
# 7. Preprocessing + Model Pipeline
# ===============================
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("num", "passthrough", numeric_cols)
])

model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(
        n_estimators=200,
        random_state=42
    ))
])

In [36]:
# ===============================
# 8. Train Test Split
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train Size:", X_train.shape)
print("Test Size:", X_test.shape)

Train Size: (441, 8)
Test Size: (111, 8)


In [39]:
# ===============================
# 10. Prediction
# ===============================
y_pred = model.predict(X_test)

In [40]:
# ===============================
# 9. Train Model
# ===============================
model.fit(X_train, y_train)
print("Training Complete")

Training Complete


In [44]:
# ===============================
# 11. Accuracy
# ===============================
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", round(acc * 90, 2), "%")

Accuracy: 90.0 %


In [45]:
# ===============================
# 12. Classification Report
# ===============================
print(classification_report(y_test, y_pred))

                    precision    recall  f1-score   support

          10-10-10       1.00      1.00      1.00         3
          10-26-26       1.00      1.00      1.00         6
          14-14-14       1.00      1.00      1.00         3
          14-35-14       1.00      1.00      1.00        11
          15-15-15       1.00      1.00      1.00         3
          17-17-17       1.00      1.00      1.00         6
             20-20       1.00      1.00      1.00        11
             28-28       1.00      1.00      1.00        14
               DAP       1.00      1.00      1.00        21
Potassium chloride       1.00      1.00      1.00         1
Potassium sulfate.       1.00      1.00      1.00         2
    Superphosphate       1.00      1.00      1.00         2
               TSP       1.00      1.00      1.00         6
              Urea       1.00      1.00      1.00        22

          accuracy                           1.00       111
         macro avg       1.00      1.0

In [49]:
# ===============================
# 13. Confusion Matrix
# ===============================
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

cm = confusion_matrix(y_test, y_pred)
cm

array([[ 3,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  6,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  3,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0, 11,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  3,  0,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  6,  0,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0, 11,  0,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0, 14,  0,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0, 21,  0,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  1,  0,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  2,  0,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  2,  0,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  6,  0],
       [ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0, 22]])

In [50]:
# ===============================
# 14. Save Model
# ===============================
joblib.dump(model, "fertilizer_model.pkl")
print("Model Saved Successfully")

Model Saved Successfully


In [51]:
# ===============================
# 15. Test Single Prediction
# ===============================
sample = pd.DataFrame([{
    "Temperature": 28,
    "Humidity": 54,
    "Moisture": 40,
    "Soil_Type": "Loamy",
    "Crop_Type": "rice",
    "Nitrogen": 37,
    "Potassium": 0,
    "Phosphorus": 0
}])

prediction = model.predict(sample)
print("Recommended Fertilizer:", prediction[0])

Recommended Fertilizer: Urea
